In [1]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'qiskit==2.2.3', 'numpy==2.2.6', 'scipy==1.15.3'])

0

In [2]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# E(x)=x0+2*x1+4*(x0+x1-1)^2
# 변수 순서 x0,x1 / 상태벡터 순서 00,01,10,11
bits = np.array([[0,0],[1,0],[0,1],[1,1]])
energy = np.array([4,1,2,7], dtype=float)
constant, a, b = 4.0, np.array([-3.,-2.]), 8.0
offset = constant + a.sum()/2 + b/4
h = -a/2 - b/4
jzz = b/4
z = 1-2*bits
assert np.allclose(energy, offset+z@h+jzz*z[:,0]*z[:,1])
gamma = np.pi/4  # 감마: 비용 위상 각도, 라디안
beta = np.pi/8  # 베타: 믹서 각도, 라디안
qc = QuantumCircuit(2)
qc.h([0,1])
initial = Statevector.from_instruction(qc)
qc.global_phase = -gamma*offset
qc.rz(2*gamma*h[0],0)
qc.rz(2*gamma*h[1],1)ㄴ
qc.rzz(2*gamma*jzz,0,1)
phase = Statevector.from_instruction(qc)
assert np.allclose(phase.data, initial.data*np.exp(-1j*gamma*energy))
qc.rx(2*beta,[0,1])
final = Statevector.from_instruction(qc)
print('QUBO 에너지:', energy.tolist())
print('Ising 상수/h/J:', offset, h.tolist(), jzz)
print('초기 진폭:', np.round(initial.data,4))
print('비용 회로 후 진폭:', np.round(phase.data,4))
print('비용 회로 후 확률:', np.round(phase.probabilities(),4))
print('믹서 후 진폭:', np.round(final.data,4))
print('믹서 후 확률:', np.round(final.probabilities(),4))
print('기대 에너지:', round(float(final.probabilities()@energy),6))
assert np.isclose(final.probabilities().sum(),1)

QUBO 에너지: [4.0, 1.0, 2.0, 7.0]
Ising 상수/h/J: 3.5 [-0.5, -1.0] 2.0
초기 진폭: [0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j]
비용 회로 후 진폭: [-0.5   +0.j      0.3536-0.3536j -0.    -0.5j     0.3536+0.3536j]
비용 회로 후 확률: [0.25 0.25 0.25 0.25]
믹서 후 진폭: [-0.7803-0.1768j  0.4268-0.1768j  0.0732-0.3232j  0.0732+0.1768j]
믹서 후 확률: [0.6402 0.2134 0.1098 0.0366]
기대 에너지: 3.25
